In [1]:
import pandas as pd

from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score




df = pd.read_csv("new_pose_detection/test_video.csv")

# Remove extra saved index column if present
df = df.drop(columns=["Unnamed: 0"], errors="ignore")




data = df.iloc[800:1801].copy()
data = data.reset_index(drop=True)




angle_cols = [
    column
    for column in data.columns
    if column.startswith("angle_")
    and column.endswith("_deg")
]




data["LDF_smooth"] = (
    data["LDF_bloodflow"]
    .rolling(
        window=5,
        min_periods=1
    )
    .mean()
)




dt = data["time_sec"].diff().median()



lag_split_index = int(len(data) * 0.80)

lag_training_data = data.iloc[
    :lag_split_index
].copy()


# Search 6–9 seconds
MIN_LAG = 60
MAX_LAG = 90

lag_results = []

for angle in angle_cols:

    for lag in range(MIN_LAG, MAX_LAG + 1):

        lagged_angle = (
            lag_training_data[angle]
            .shift(lag)
        )

        correlation = lagged_angle.corr(
            lag_training_data["LDF_smooth"]
        )

        if pd.notna(correlation):

            lag_results.append({
                "Angle": angle,
                "Lag_samples": lag,
                "Correlation": correlation,
                "Absolute_correlation": abs(correlation)
            })


lag_results_df = pd.DataFrame(lag_results)

if lag_results_df.empty:
    raise ValueError(
        "No valid lag correlations were calculated."
    )


# ============================================================
# 7. FIND STRONGEST ANGLE AND LAG
# ============================================================

best_lag_per_angle = (
    lag_results_df
    .loc[
        lag_results_df
        .groupby("Angle")["Absolute_correlation"]
        .idxmax()
    ]
    .sort_values(
        "Absolute_correlation",
        ascending=False
    )
    .reset_index(drop=True)
)

best_result = best_lag_per_angle.iloc[0]

best_angle = best_result["Angle"]
best_lag = int(best_result["Lag_samples"])


# Convert angle_3_deg into the number 3
angle_number = int(
    best_angle
    .replace("angle_", "")
    .replace("_deg", "")
)




angular_velocity_col = (
    f"angular_velocity_{angle_number}_deg_per_s"
)

if angular_velocity_col not in data.columns:
    raise ValueError(
        f"Column not found: {angular_velocity_col}"
    )


vertex_speed_matches = [
    column
    for column in data.columns
    if column.startswith(
        f"vertex_speed_angle_{angle_number}_"
    )
]

if len(vertex_speed_matches) == 0:
    raise ValueError(
        f"No vertex-speed column found for {best_angle}."
    )

vertex_speed_col = vertex_speed_matches[0]


print("Angle:", best_angle)
print("Angular velocity:", angular_velocity_col)
print("Vertex speed:", vertex_speed_col)
print("Lag:", best_lag, "samples")
print("Lag:", round(best_lag * dt, 2), "seconds")




lagged_angle_col = f"{best_angle}_lagged"

lagged_velocity_col = (
    f"{angular_velocity_col}_lagged"
)

lagged_vertex_speed_col = (
    f"{vertex_speed_col}_lagged"
)


data[lagged_angle_col] = (
    data[best_angle].shift(best_lag)
)

data[lagged_velocity_col] = (
    data[angular_velocity_col].shift(best_lag)
)

data[lagged_vertex_speed_col] = (
    data[vertex_speed_col].shift(best_lag)
)




feature_cols = [
    lagged_angle_col,
    lagged_velocity_col,
    lagged_vertex_speed_col
]

model_data = data[
    feature_cols + ["LDF_smooth"]
].dropna().reset_index(drop=True)

if len(model_data) < 20:
    raise ValueError(
        "Too few complete rows remain after lagging."
    )




split_index = int(len(model_data) * 0.80)

train_data = model_data.iloc[:split_index]
test_data = model_data.iloc[split_index:]

X_train = train_data[feature_cols]
X_test = test_data[feature_cols]

y_train = train_data["LDF_smooth"]
y_test = test_data["LDF_smooth"]




# SVR is sensitive to feature scales, so StandardScaler is
# included inside a pipeline. It fits only on the training data.
support_vector_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "svr",
        SVR(
            kernel="rbf",
            C=10,
            epsilon=0.1,
            gamma="scale"
        )
    )
])

support_vector_model.fit(
    X_train,
    y_train
)




predictions = support_vector_model.predict(
    X_test
)

mae = mean_absolute_error(
    y_test,
    predictions
)

r2 = r2_score(
    y_test,
    predictions
)

print("\nMAE:", mae)
print("R²:", r2)

Angle: angle_3_deg
Angular velocity: angular_velocity_3_deg_per_s
Vertex speed: vertex_speed_angle_3_p13_coord_per_s
Lag: 77 samples
Lag: 7.7 seconds

MAE: 33.42724751953423
R²: 0.3116197442261235
